# FLEO — FER2013 deltas (recovered weights) + RAF-DB 100 ep

**Settings:** Accelerator = **GPU T4 x2**, Internet = **ON**.
**Add Input (3):** `msambare/fer2013`, `shuvoalok/raf-db-dataset`, **and your weights
dataset** (upload the folder `kaggle_weights_upload` containing `baseline_best.pt`
and `fleo_best.pt` as a private Kaggle Dataset, then attach it).

Run with **Save Version -> Save & Run All (Commit)**.

Plan (total ~5 h, fits the 12 h wall with a large margin):
1. FER2013 **Delta_fold + accuracy + macro-F1** computed from the recovered
   100-epoch weights — **no retraining**.
2. Zip the FER2013 numbers immediately.
3. RAF-DB baseline + FLEO at **100 epochs** (time-guarded: skipped cleanly if
   less than 5 h of budget remain), then its deltas + zip.

## 1. Clone + install

In [ ]:
import time, pathlib
pathlib.Path('/kaggle/working/t0.txt').write_text(str(time.time()))
%cd /kaggle/working
!rm -rf FLEO
!git clone https://github.com/olfa-askri/FLEO.git
%cd /kaggle/working/FLEO
!pip install -q ultralytics onnx onnxruntime onnxscript
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Auto-find datasets + recovered weights

In [ ]:
import glob, os, pathlib
def find_root(keys):
    for p in sorted(glob.glob('/kaggle/input/*'))+sorted(glob.glob('/kaggle/input/*/*'))+sorted(glob.glob('/kaggle/input/*/*/*')):
        if os.path.isdir(p) and any(k in p.lower() for k in keys):
            return p
    return None
fer = find_root(['fer2013','fer-2013'])
raf = find_root(['raf-db','rafdb','raf_db'])
hits = {p.name: str(p) for p in pathlib.Path('/kaggle/input').rglob('*_best.pt')}
WF = hits.get('fleo_best.pt'); WB = hits.get('baseline_best.pt')
print('FER root:', fer); print('RAF root:', raf)
print('FLEO weights:', WF); print('baseline weights:', WB)
assert fer and raf, 'Attach msambare/fer2013 and shuvoalok/raf-db-dataset via Add Input!'
assert WF and WB, 'Attach your weights dataset (fleo_best.pt + baseline_best.pt) via Add Input!'

In [ ]:
!python -m data.prepare_fer2013 --src "{fer}" --out /kaggle/working/FLEO/datasets/fer2013
!python -m data.prepare_rafdb   --src "{raf}" --out /kaggle/working/FLEO/datasets/rafdb
!ls -l /kaggle/working/FLEO/datasets/fer2013/data.yaml /kaggle/working/FLEO/datasets/rafdb/data.yaml

## 3. FER2013 — Delta_fold + accuracy + macro-F1 from the recovered weights (~40 min, no retraining)

In [ ]:
!python -m scripts.deltas --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 --imgsz 160 --device 0 --weights {WF} --out /kaggle/working/FLEO/results/deltas_fer2013.json
!python -m scripts.deltas --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --dataset fer2013 --imgsz 160 --device 0 --weights {WF} --metric macro_f1 --out /kaggle/working/FLEO/results/macro_f1_fer2013.json
!python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/fer2013/data.yaml --weights {WB} --imgsz 160 --device 0 --out /kaggle/working/FLEO/results/baseline_fer2013.json

## 4. Zip the FER2013 numbers NOW

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_fer2013_deltas.zip results
!ls -lh /kaggle/working/fleo_fer2013_deltas.zip

## 5. RAF-DB — baseline + FLEO, 100 epochs (time-guarded)

In [ ]:
import time, pathlib
t0 = float(pathlib.Path('/kaggle/working/t0.txt').read_text())
left_h = 11.5 - (time.time() - t0) / 3600
print('Budget left before the 12h wall (0.5h safety margin): %.2f h' % left_h)
if left_h > 5.0:
    !python -m scripts.run_matrix --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --dataset rafdb --seeds 0 --epochs 100 --imgsz 160 --batch 64 --device 0 --project /kaggle/working/FLEO/runs/rafdb
else:
    print('SKIPPED RAF-DB: not enough time left. FER2013 deltas are zipped and safe; rerun RAF-DB in a fresh session.')

## 6. RAF-DB — the numbers (skipped gracefully if the guard fired)

In [ ]:
import json, os
p = '/kaggle/working/FLEO/results/deltas_rafdb.json'
if os.path.exists(p):
    print('=== RAF-DB Delta_fold + accuracy ===')
    print(json.dumps(json.load(open(p)), indent=2))
    WFr = '/kaggle/working/FLEO/runs/rafdb/fleo_seed0/weights/best.pt'
    WBr = '/kaggle/working/FLEO/runs/rafdb/baseline_seed0/weights/best.pt'
    print('\n=== FLEO macro-F1 (RAF-DB) ===')
    !python -m scripts.deltas --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --dataset rafdb --imgsz 160 --device 0 --weights {WFr} --metric macro_f1 --out /kaggle/working/FLEO/results/macro_f1_rafdb.json
    print('\n=== BASELINE (RAF-DB) ===')
    !python -m scripts.evaluate --data /kaggle/working/FLEO/datasets/rafdb/data.yaml --weights {WBr} --imgsz 160 --device 0 --out /kaggle/working/FLEO/results/baseline_rafdb.json
else:
    print('RAF-DB was skipped by the time guard - no numbers this run.')

## 7. Final bundle

In [ ]:
%cd /kaggle/working/FLEO
!zip -qr /kaggle/working/fleo_all.zip runs results
!ls -lh /kaggle/working/*.zip
print('DONE. Download the zips from the Output panel once the commit finishes.')